# 🛒 Olist E-commerce Analytics
## Descoberta dos Dados e Avaliação Inicial da Qualidade

**Projeto:** Análise de Dados de E-commerce Brasileiro  
**Dataset:** Olist Brazilian E-commerce Public Dataset  
**Etapa:** 01 — Data Discovery  
**Autor:** João Victor Azevedo Porto

---

## 📌 Contexto do Projeto

Este projeto tem como objetivo analisar dados de um marketplace brasileiro para entender quatro áreas principais do negócio:

- desempenho das vendas;
- comportamento dos clientes;
- eficiência das entregas;
- satisfação dos consumidores.

Antes de criar consultas SQL, dashboards ou indicadores, precisamos primeiro entender profundamente os dados disponíveis.

Essa etapa é chamada de **Data Discovery**, ou **Descoberta dos Dados**.

> **Nota da revisão:** os textos e o código foram revisados a partir deste arquivo e de suas saídas salvas. Os CSVs não estão anexados e o conjunto completo não foi reexecutado nesta revisão. Saídas de células alteradas foram removidas; saídas de células inalteradas são as do original. Execute **Ambiente de execução → Executar tudo** no Colab para atualizar os resultados. As verificações adicionais não modificam os dados brutos nem arquivos externos.

# 🎯 Objetivos desta etapa

Ao final deste notebook, queremos conseguir responder:

1. Quantos arquivos e tabelas existem no dataset?
2. Quantas linhas e colunas existem em cada tabela?
3. Quais são os tipos de dados de cada coluna?
4. Existem valores nulos?
5. Existem registros duplicados?
6. Quais colunas podem funcionar como chaves?
7. Como as tabelas se relacionam?
8. Qual período de tempo os dados cobrem?
9. Existem valores estranhos ou inconsistentes?
10. O que precisará ser tratado na próxima etapa de limpeza?



# 1. Preparação do ambiente

## O que vamos fazer?

Primeiro importamos as bibliotecas necessárias.

Neste momento utilizaremos:

### Pandas

Biblioteca principal para manipulação e análise de dados em Python.

### NumPy

Biblioteca muito utilizada em cálculos numéricos.


### Pathlib

Biblioteca utilizada para trabalhar com caminhos de arquivos e pastas de forma mais segura.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
# Formatação afeta somente a exibição, não os valores nem seus tipos.
# Evitamos arredondar globalmente as coordenadas geográficas.
print("Versão do pandas:", pd.__version__)

Versão do pandas: 2.2.3


# 2. Download do conjunto de dados no Google Colab

Usaremos **KaggleHub** para baixar os arquivos do conjunto Olist e identificar os CSVs automaticamente, sem `files.upload()` e sem montar o Google Drive.

O download depende de acesso ao Kaggle. Para reproduzir a análise, registre a versão do conjunto e as versões das bibliotecas usadas.

In [2]:
%pip install -q kagglehub

import kagglehub
from importlib.metadata import version

print("Versão do KaggleHub:", version("kagglehub"))

Versão do KaggleHub: 1.0.2


## 2.1 Fazendo o download do dataset Olist

Agora vamos baixar o dataset diretamente do Kaggle.

A função:

```python
kagglehub.dataset_download()
```

recebe o identificador do dataset no Kaggle e retorna o caminho onde os arquivos foram salvos no ambiente do Colab.

Para o dataset Olist, utilizaremos:

```text
olistbr/brazilian-ecommerce
```


In [3]:
# Download automático do dataset
dataset_path = kagglehub.dataset_download(
    "olistbr/brazilian-ecommerce"
)

# Convertendo em um objeto Path
RAW_DATA_PATH = Path(dataset_path)

print('Dataset baixado com sucesso')
print('Caminho dos dados', RAW_DATA_PATH)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Dataset baixado com sucesso
Caminho dos dados /kaggle/input/brazilian-ecommerce


# 3. Conferindo os arquivos baixados

Agora vamos verificar se os arquivos CSV do dataset foram realmente encontrados.

Essa etapa confirma:

- quantos CSVs estão disponíveis;
- quais são seus nomes;
- se o download foi realizado corretamente.


In [4]:
csv_files = sorted(RAW_DATA_PATH.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"Nenhum arquivo CSV encontrado em {RAW_DATA_PATH}.")
print(f"Quantidade de arquivos CSV encontrados: {len(csv_files)}")
for file in csv_files:
    print("•", file.name)

Quantidade de arquivos CSV encontrados: 9
• olist_customers_dataset.csv
• olist_geolocation_dataset.csv
• olist_order_items_dataset.csv
• olist_order_payments_dataset.csv
• olist_order_reviews_dataset.csv
• olist_orders_dataset.csv
• olist_products_dataset.csv
• olist_sellers_dataset.csv
• product_category_name_translation.csv


# 4. Carregando os datasets

Agora vamos carregar todos os CSVs utilizando Pandas.

Normalmente poderíamos fazer:

```python
df = pd.read_csv("arquivo.csv")
```

Mas como temos vários arquivos, vamos automatizar o processo.

Cada arquivo será armazenado dentro de um **dicionário Python**.


In [5]:
datasets = {}

for file in csv_files:

  table_name = file.stem

  datasets[table_name] = pd.read_csv(file)

print(f'Quantidade de tabelas carregadas: {len(datasets)}')

Quantidade de tabelas carregadas: 9


# 5. Criando um inventário dos datasets

Vamos descobrir:

- nome da tabela;
- quantidade de linhas;
- quantidade de colunas.

Esse é nosso primeiro panorama do banco de dados.


In [6]:
inventario = (
    pd.DataFrame([
        {"tabela": name, "linhas": len(df), "colunas": df.shape[1]}
        for name, df in datasets.items()
    ])
    .sort_values("linhas", ascending=False)
    .reset_index(drop=True)
)
inventario

,tabela,linhas,colunas
0,olist_geolocation_dataset,1000163,5
1,olist_order_items_dataset,112650,7
2,olist_order_payments_dataset,103886,5
3,olist_customers_dataset,99441,5
4,olist_orders_dataset,99441,8
5,olist_order_reviews_dataset,99224,7
6,olist_products_dataset,32951,9
7,olist_sellers_dataset,3095,4
8,product_category_name_translation,71,2


# 6. Conhecendo o schema de cada tabela

Agora vamos visualizar:

- nome das colunas;
- tipos de dados;
- primeiras linhas.




In [7]:
for name, df in datasets.items():

  print('=' * 100)
  print(f'Tabela: {name}')
  print('=' * 100)

  print('\nTipos das colunas:')
  print(df.dtypes)

  print('\nPrimeiras linhas:')
  display(df.head())

  print()


Tabela: olist_customers_dataset

Tipos das colunas:
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

Primeiras linhas:


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP



Tabela: olist_geolocation_dataset

Tipos das colunas:
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

Primeiras linhas:


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP



Tabela: olist_order_items_dataset

Tipos das colunas:
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

Primeiras linhas:


...

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0



Tabela: olist_sellers_dataset

Tipos das colunas:
seller_id                 object
seller_zip_code_prefix     int64
seller_city               object
seller_state              object
dtype: object

Primeiras linhas:


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP



Tabela: product_category_name_translation

Tipos das colunas:
product_category_name            object
product_category_name_english    object
dtype: object

Primeiras linhas:


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


# 7. Avaliando os tipos de dados

Agora vamos construir uma tabela consolidada contendo o tipo de cada coluna em cada dataset.

Isso facilitará a identificação de colunas que precisam ser convertidas depois.


In [8]:
dtype_sumario = []

for name, df in datasets.items():

  for coluna in df.columns:

    dtype_sumario.append({
        'tabela' : name,
        'coluna' : coluna,
        'tipo' : str(df[coluna].dtype)
    })

dtype_sumario = pd.DataFrame(dtype_sumario)

dtype_sumario

,tabela,coluna,tipo
0,olist_customers_dataset,customer_id,object
1,olist_customers_dataset,customer_unique_id,object
2,olist_customers_dataset,customer_zip_code_prefix,int64
3,olist_customers_dataset,customer_city,object
4,olist_customers_dataset,customer_state,object
5,olist_geolocation_dataset,geolocation_zip_code_prefix,int64
6,olist_geolocation_dataset,geolocation_lat,float64
7,olist_geolocation_dataset,geolocation_lng,float64
8,olist_geolocation_dataset,geolocation_city,object
9,olist_geolocation_dataset,geolocation_state,object


## 7.1 Tipos recomendados para a camada tratada

| Campos | Tipo atual | Tipo recomendado e condição |
|---|---|---|
| As oito datas listadas na seção 12 | `object` | `datetime64[ns]`, separando nulos originais de falhas de conversão |
| `product_photos_qty`, `product_name_lenght`, `product_description_lenght` | `float64` | `Int64` após validar que os preenchidos são finitos, inteiros e não negativos |
| Prefixos de CEP de clientes, vendedores e geolocalização | `int64` | `string` com cinco dígitos; preferir especificar o tipo ao ler o CSV |
| IDs, cidades e comentários | `object` | `string` para semântica explícita; `object` textual não é, por si só, um erro |
| Status, UF, tipo de pagamento e categoria | `object` | `string`; considerar `category` após padronizar os valores |
| Preço, frete e valor de pagamento | `float64` | Manter decimais para exploração; usar centavos inteiros ou `NUMERIC` no SQL para reconciliação exata |
| Peso, dimensões, latitude e longitude | `float64` | Manter decimais; uma medida não precisa virar inteiro porque a amostra parece inteira |
| Nota, parcelas e sequenciais | `int64` | Manter inteiros e validar o domínio de cada campo |

É comum uma contagem aparecer como `float64` quando a inferência de leitura precisa representar `NaN`. O tipo `Int64` (I maiúsculo) aceita inteiros e ausentes. **Não use arredondamento para esconder frações inesperadas.**

Na limpeza, preserve os originais e faça as conversões em uma cópia. Os tipos numéricos dos prefixos são mantidos nesta descoberta para documentar a inferência original.

In [9]:
products_type_check = datasets["olist_products_dataset"]
integer_checks = []
for col in ["product_photos_qty", "product_name_lenght", "product_description_lenght"]:
    raw = products_type_check[col]
    numeric = pd.to_numeric(raw, errors="coerce")
    finite = numeric.notna() & np.isfinite(numeric)
    fractions = finite & numeric.mod(1).ne(0)
    integer_checks.append({
        "coluna": col,
        "nulos_originais": int(raw.isna().sum()),
        "nao_numericos": int((raw.notna() & numeric.isna()).sum()),
        "nao_finitos": int((numeric.notna() & ~np.isfinite(numeric)).sum()),
        "fracionarios": int(fractions.sum()),
        "negativos": int(numeric.lt(0).sum()),
    })
display(pd.DataFrame(integer_checks))
# Só converter para Int64 na etapa de limpeza, depois de avaliar este relatório.

,coluna,nulos_originais,nao_numericos,nao_finitos,fracionarios,negativos
0,product_photos_qty,610,0,0,0,0
1,product_name_lenght,610,0,0,0,0
2,product_description_lenght,610,0,0,0,0


# 8. Análise de valores ausentes

Esta análise contabiliza ausentes reconhecidos pelo pandas, como `NaN`, `None` e `pd.NA`.

A leitura de CSV reconhece diversos marcadores de ausência, mas strings contendo apenas espaços podem permanecer como texto. Por isso, uma validação complementar verifica campos em branco.

**Ausência não é automaticamente erro:** comentários podem não ter sido preenchidos, e uma data de entrega pode não existir para um pedido cancelado ou ainda não entregue. A interpretação depende do campo e do status do pedido.

In [10]:
missing_values = []

for name, df in datasets.items():

  for coluna in df.columns:

    # Conta quantos valores estão ausentes:
    missing_count = df[coluna].isna().sum()

    # Calcula o percentual de valores ausentes
    missing_percent =(
        (missing_count / len(df)) * 100
        if len(df) else 0
  )

    # Mostramos apenas colunas que realmente possuem valores ausentes
    if missing_count > 0:
      missing_values.append({
          'tabela' : name,
          'coluna' : coluna,
          'quantidade_nulos': missing_count,
          'percentual_nulos' : round(missing_percent,2)
    })

missing_df = pd.DataFrame(missing_values)

if not missing_df.empty:
  missing_df = missing_df.sort_values(
      ['percentual_nulos', 'quantidade_nulos'],
      ascending = False
  ).reset_index(drop=True)

missing_df

,tabela,coluna,quantidade_nulos,percentual_nulos
0,olist_order_reviews_dataset,review_comment_title,87656,88.34
1,olist_order_reviews_dataset,review_comment_message,58247,58.70
2,olist_orders_dataset,order_delivered_customer_date,2965,2.98
3,olist_products_dataset,product_category_name,610,1.85
4,olist_products_dataset,product_name_lenght,610,1.85
5,olist_products_dataset,product_description_lenght,610,1.85
6,olist_products_dataset,product_photos_qty,610,1.85
7,olist_orders_dataset,order_delivered_carrier_date,1783,1.79
8,olist_orders_dataset,order_approved_at,160,0.16
9,olist_products_dataset,product_weight_g,2,0.01


## 8.1 Campos em branco e ausentes por status

Esta verificação distingue texto em branco de ausentes já reconhecidos e mede datas ausentes dentro de cada status. Uma entrega ausente em um pedido entregue exige investigação diferente de uma entrega ausente em um pedido cancelado.

In [11]:
blank_records = []
for name, df in datasets.items():
    for col in df.select_dtypes(include=["object", "string"]).columns:
        blank_count = int(df[col].astype("string").str.strip().eq("").fillna(False).sum())
        if blank_count:
            blank_records.append({"tabela": name, "coluna": col, "em_branco": blank_count})
display(pd.DataFrame(blank_records, columns=["tabela", "coluna", "em_branco"]))
orders_missing = datasets["olist_orders_dataset"]
status_records = []
for status, group in orders_missing.groupby("order_status", dropna=False):
    for col in ["order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"]:
        status_records.append({
            "status": status, "coluna": col, "pedidos": len(group),
            "nulos": int(group[col].isna().sum()),
            "percentual_no_status": round(group[col].isna().mean() * 100, 2),
        })
display(pd.DataFrame(status_records))

,tabela,coluna,em_branco
0,olist_order_reviews_dataset,review_comment_title,2
1,olist_order_reviews_dataset,review_comment_message,27


,status,coluna,pedidos,nulos,percentual_no_status
0,approved,order_approved_at,2,0,0.00
1,approved,order_delivered_carrier_date,2,2,100.00
2,approved,order_delivered_customer_date,2,2,100.00
3,canceled,order_approved_at,625,141,22.56
4,canceled,order_delivered_carrier_date,625,550,88.00
5,canceled,order_delivered_customer_date,625,619,99.04
6,created,order_approved_at,5,5,100.00
7,created,order_delivered_carrier_date,5,5,100.00
8,created,order_delivered_customer_date,5,5,100.00
9,delivered,order_approved_at,96478,14,0.01


# 9. Verificando registros duplicados

Agora vamos verificar se existem linhas exatamente iguais dentro das tabelas.

Duplicatas podem causar problemas como:

- contagem duplicada de pedidos;
- faturamento duplicado;
- indicadores incorretos.


In [12]:
duplicate_sumario = pd.DataFrame([
    {
        'tabela' : name,
        'linhas_duplicadas' : df.duplicated().sum(),
        'percentual_duplicado' : round(
            df.duplicated().sum() / len(df) * 100,
            4
        ) if len(df) else 0
    }
    for name, df in datasets.items()
])

duplicate_sumario



,tabela,linhas_duplicadas,percentual_duplicado
0,olist_customers_dataset,0,0.0000
1,olist_geolocation_dataset,261831,26.1788
2,olist_order_items_dataset,0,0.0000
3,olist_order_payments_dataset,0,0.0000
4,olist_order_reviews_dataset,0,0.0000
5,olist_orders_dataset,0,0.0000
6,olist_products_dataset,0,0.0000
7,olist_sellers_dataset,0,0.0000
8,product_category_name_translation,0,0.0000


# 10. Identificando possíveis chaves

Em bancos de dados, uma **chave primária** identifica cada registro de forma única.

Exemplo:

```text
order_id
```

na tabela de pedidos.

Para investigar isso, vamos verificar:

- quantidade de linhas;
- quantidade de valores únicos;
- presença de nulos;
- se a coluna é realmente única.


In [13]:
def uniqueness_report(df, colunas):

  records = []

  for coluna in colunas:

    records.append({
        'coluna' : coluna,
        'quantidade_linhas' : len(df),
        'valores_unicos' : df[coluna].nunique(dropna=False),
        'eh_unico' : df[coluna].is_unique,
        'valores_nulos' : df[coluna].isna().sum()
    })

  return pd.DataFrame(records)

## 10.1 Clientes

Vamos analisar:

```text
customer_id
customer_unique_id
```

Eles possuem funções diferentes.

- `customer_id` → identifica o cliente dentro de um pedido;
- `customer_unique_id` → ajuda a identificar o mesmo consumidor em compras diferentes.


In [14]:
customers_name = next(
    (name for name in datasets if "customers" in name),
    None
)

if customers_name:

  customers = datasets[customers_name]

  display(
      uniqueness_report(
          customers,
          ['customer_id', 'customer_unique_id'])
  )

,coluna,quantidade_linhas,valores_unicos,eh_unico,valores_nulos
0,customer_id,99441,99441,True,0
1,customer_unique_id,99441,96096,False,0


### 🔎 O que foi observado?

`customer_id` é único, logo, ele pode funcionar como chave da tabela de clientes.

Já `customer_unique_id` tem valores repetidos, porque o mesmo consumidor pode aparecer em mais de um pedido.


## 10.2 Pedidos

Agora vamos analisar a tabela de pedidos.

Esperamos que:

```text
order_id
```

seja único.


In [15]:
orders_name = next(
    (name for name in datasets if name.endswith('orders_dataset')),
    None
)

if orders_name:

  orders = datasets[orders_name]

  display(
      uniqueness_report(
          orders,
          ['order_id', 'customer_id']
        )
  )

,coluna,quantidade_linhas,valores_unicos,eh_unico,valores_nulos
0,order_id,99441,99441,True,0
1,customer_id,99441,99441,True,0


## 10.3 Itens dos pedidos

Aqui existe uma diferença importante.

Um pedido pode possuir vários itens.

Portanto:

```text
order_id
```

não precisa ser único nessa tabela.

Uma possível chave composta é:

```text
order_id + order_item_id
```


In [16]:
items_name = next(
    (name for name in datasets if 'order_items' in name),
    None
)

if items_name:

  order_items = datasets[items_name]

  display(
      uniqueness_report(
          order_items,
          ['order_id', 'product_id', 'seller_id']
        )
  )

  if {'order_id', 'order_item_id'}.issubset(order_items.columns):

    print(
        'Duplicatas na chave composta(order_id + order_item_id):',
        order_items.duplicated(
        subset=['order_id', 'order_item_id']
    ).sum()
    )

,coluna,quantidade_linhas,valores_unicos,eh_unico,valores_nulos
0,order_id,112650,98666,False,0
1,product_id,112650,32951,False,0
2,seller_id,112650,3095,False,0


Duplicatas na chave composta(order_id + order_item_id): 0


# 11. Validação dos relacionamentos entre tabelas

Vamos verificar se as chaves das tabelas dependentes existem nas tabelas de referência. Por exemplo, cada `customer_id` de pedidos deve encontrar correspondência em clientes.

A função abaixo conta **chaves distintas sem correspondência**, não o número de linhas afetadas. As verificações complementares ao final da seção apresentam ambas as medidas e os nulos separadamente.

In [17]:
def unmatched_keys(child_df, child_key, parent_df, parent_key):

  child_values = set(
      child_df[child_key].dropna().unique()
  )

  parent_values = set(
      parent_df[parent_key].dropna().unique()
  )

  # Retorna quantas existem na tabela filha mas não na tabela pai
  return len(child_values - parent_values)

## 11.1 Pedidos → Clientes


In [18]:
if customers_name and orders_name:
    print(
        "Chaves customer_id distintas em pedidos sem correspondência em clientes:",
        unmatched_keys(datasets[orders_name], "customer_id",
                       datasets[customers_name], "customer_id")
    )

Chaves customer_id distintas em pedidos sem correspondência em clientes: 0


## 11.2 Itens → Pedidos


In [19]:
if orders_name and items_name:
    print(
        "Chaves order_id distintas em itens sem correspondência em pedidos:",
        unmatched_keys(datasets[items_name], "order_id",
                       datasets[orders_name], "order_id")
    )

Chaves order_id distintas em itens sem correspondência em pedidos: 0


## 11.3 Cobertura das relações e chaves adicionais

O relatório distingue nulos, linhas sem correspondência e chaves distintas sem correspondência. Relações com geolocalização medem cobertura; não pressupõem que cada prefixo seja único.

Uma chave primária exige unicidade **e** ausência de nulos. A chave de pagamentos é candidata até a execução desta verificação. Não assuma unicidade de `review_id`.

In [20]:
relations = [
    ("olist_orders_dataset", "customer_id", "olist_customers_dataset", "customer_id"),
    ("olist_order_items_dataset", "order_id", "olist_orders_dataset", "order_id"),
    ("olist_order_items_dataset", "product_id", "olist_products_dataset", "product_id"),
    ("olist_order_items_dataset", "seller_id", "olist_sellers_dataset", "seller_id"),
    ("olist_order_payments_dataset", "order_id", "olist_orders_dataset", "order_id"),
    ("olist_order_reviews_dataset", "order_id", "olist_orders_dataset", "order_id"),
    ("olist_products_dataset", "product_category_name", "product_category_name_translation", "product_category_name"),
    ("olist_customers_dataset", "customer_zip_code_prefix", "olist_geolocation_dataset", "geolocation_zip_code_prefix"),
    ("olist_sellers_dataset", "seller_zip_code_prefix", "olist_geolocation_dataset", "geolocation_zip_code_prefix"),
]
relation_records = []
for child_name, child_key, parent_name, parent_key in relations:
    child = datasets[child_name][child_key]
    parent = datasets[parent_name][parent_key]
    orphan = child.notna() & ~child.isin(parent.dropna())
    relation_records.append({
        "origem": child_name, "chave": child_key, "destino": parent_name,
        "nulos_na_origem": int(child.isna().sum()),
        "linhas_sem_correspondencia": int(orphan.sum()),
        "chaves_distintas_sem_correspondencia": int(child[orphan].nunique()),
        "chave_destino_unica_sem_nulos": bool(parent.is_unique and parent.notna().all()),
    })
display(pd.DataFrame(relation_records))

key_records = []
for name, keys in [
    ("olist_order_items_dataset", ["order_id", "order_item_id"]),
    ("olist_order_payments_dataset", ["order_id", "payment_sequential"]),
    ("olist_order_reviews_dataset", ["review_id"]),
    ("olist_order_reviews_dataset", ["review_id", "order_id"]),
    ("olist_products_dataset", ["product_id"]),
    ("olist_sellers_dataset", ["seller_id"]),
    ("product_category_name_translation", ["product_category_name"]),
]:
    df = datasets[name]
    null_rows = int(df[keys].isna().any(axis=1).sum())
    duplicates = int(df.duplicated(keys).sum())
    key_records.append({"tabela": name, "chave": " + ".join(keys),
                        "linhas_com_nulos": null_rows, "duplicatas_excedentes": duplicates,
                        "candidata_valida_na_amostra": null_rows == 0 and duplicates == 0})
display(pd.DataFrame(key_records))

coverage = []
order_ids = datasets["olist_orders_dataset"]["order_id"]
for name in ["olist_order_items_dataset", "olist_order_payments_dataset", "olist_order_reviews_dataset"]:
    counts = datasets[name].groupby("order_id").size()
    coverage.append({"tabela": name,
                     "pedidos_sem_registro": int((~order_ids.isin(counts.index)).sum()),
                     "pedidos_com_varias_linhas": int(counts.gt(1).sum()),
                     "maximo_linhas_por_pedido": int(counts.max()) if len(counts) else 0})
display(pd.DataFrame(coverage))

,origem,chave,destino,nulos_na_origem,linhas_sem_correspondencia,chaves_distintas_sem_correspondencia,chave_destino_unica_sem_nulos
0,olist_orders_dataset,customer_id,olist_customers_dataset,0,0,0,True
1,olist_order_items_dataset,order_id,olist_orders_dataset,0,0,0,True
2,olist_order_items_dataset,product_id,olist_products_dataset,0,0,0,True
3,olist_order_items_dataset,seller_id,olist_sellers_dataset,0,0,0,True
4,olist_order_payments_dataset,order_id,olist_orders_dataset,0,0,0,True
5,olist_order_reviews_dataset,order_id,olist_orders_dataset,0,0,0,True
6,olist_products_dataset,product_category_name,product_category_name_translation,610,13,2,True
7,olist_customers_dataset,customer_zip_code_prefix,olist_geolocation_dataset,0,278,157,False
8,olist_sellers_dataset,seller_zip_code_prefix,olist_geolocation_dataset,0,7,7,False


,tabela,chave,linhas_com_nulos,duplicatas_excedentes,candidata_valida_na_amostra
0,olist_order_items_dataset,order_id + order_item_id,0,0,True
1,olist_order_payments_dataset,order_id + payment_sequential,0,0,True
2,olist_order_reviews_dataset,review_id,0,814,False
3,olist_order_reviews_dataset,review_id + order_id,0,0,True
4,olist_products_dataset,product_id,0,0,True
5,olist_sellers_dataset,seller_id,0,0,True
6,product_category_name_translation,product_category_name,0,0,True


,tabela,pedidos_sem_registro,pedidos_com_varias_linhas,maximo_linhas_por_pedido
0,olist_order_items_dataset,775,9803,21
1,olist_order_payments_dataset,1,2961,29
2,olist_order_reviews_dataset,768,547,3


# 12. Período dos dados e qualidade das datas

Usamos uma lista explícita das **oito colunas temporais**. A busca apenas por `date` e `timestamp` deixava `order_approved_at` de fora.

Cada evento tem seu próprio período. O intervalo das compras não representa necessariamente os limites de todas as datas do conjunto.

In [21]:
date_columns = [
    ("olist_order_items_dataset", "shipping_limit_date"),
    ("olist_order_reviews_dataset", "review_creation_date"),
    ("olist_order_reviews_dataset", "review_answer_timestamp"),
    ("olist_orders_dataset", "order_purchase_timestamp"),
    ("olist_orders_dataset", "order_approved_at"),
    ("olist_orders_dataset", "order_delivered_carrier_date"),
    ("olist_orders_dataset", "order_delivered_customer_date"),
    ("olist_orders_dataset", "order_estimated_delivery_date"),
]
date_columns

[('olist_order_items_dataset', 'shipping_limit_date'),
 ('olist_order_reviews_dataset', 'review_creation_date'),
 ('olist_order_reviews_dataset', 'review_answer_timestamp'),
 ('olist_orders_dataset', 'order_purchase_timestamp'),
 ('olist_orders_dataset', 'order_approved_at'),
 ('olist_orders_dataset', 'order_delivered_carrier_date'),
 ('olist_orders_dataset', 'order_delivered_customer_date'),
 ('olist_orders_dataset', 'order_estimated_delivery_date')]

As datas são convertidas **apenas para diagnóstico**, sem modificar `datasets`.

O formato esperado é `AAAA-MM-DD HH:MM:SS`. Valores preenchidos que não respeitem esse formato são sinalizados para inspeção, sem apagar a informação original. O relatório distingue ausentes de falhas de conversão.

In [22]:
date_range_summary = []
parsed_dates = {}
for table_name, column in date_columns:
    raw = datasets[table_name][column]
    parsed = pd.to_datetime(raw, format="%Y-%m-%d %H:%M:%S", errors="coerce")
    parsed_dates[(table_name, column)] = parsed
    invalid = raw.notna() & parsed.isna()
    date_range_summary.append({
        "tabela": table_name, "coluna": column,
        "primeira_data": parsed.min(), "ultima_data": parsed.max(),
        "nulos_originais": int(raw.isna().sum()),
        "falhas_conversao": int(invalid.sum()),
        "exemplos_invalidos": raw.loc[invalid].head(3).tolist(),
    })
date_range_df = pd.DataFrame(date_range_summary)
date_range_df

,tabela,coluna,primeira_data,ultima_data,nulos_originais,falhas_conversao,exemplos_invalidos
0,olist_order_items_dataset,shipping_limit_date,2016-09-19 00:15:34,2020-04-09 22:35:08,0,0,[]
1,olist_order_reviews_dataset,review_creation_date,2016-10-02 00:00:00,2018-08-31 00:00:00,0,0,[]
2,olist_order_reviews_dataset,review_answer_timestamp,2016-10-07 18:32:28,2018-10-29 12:27:35,0,0,[]
3,olist_orders_dataset,order_purchase_timestamp,2016-09-04 21:15:19,2018-10-17 17:30:18,0,0,[]
4,olist_orders_dataset,order_approved_at,2016-09-15 12:16:38,2018-09-03 17:40:06,160,0,[]
5,olist_orders_dataset,order_delivered_carrier_date,2016-10-08 10:34:01,2018-09-11 19:48:28,1783,0,[]
6,olist_orders_dataset,order_delivered_customer_date,2016-10-11 13:46:32,2018-10-17 13:22:46,2965,0,[]
7,olist_orders_dataset,order_estimated_delivery_date,2016-09-30 00:00:00,2018-11-12 00:00:00,0,0,[]


## 12.1 Inspeção dos prazos extremos

A saída original mostra `shipping_limit_date` até 09/04/2020. A tabela abaixo apresenta os maiores prazos e seu contexto. Ela não considera automaticamente inválido todo prazo posterior à última compra.

In [23]:
shipping_check = datasets["olist_order_items_dataset"][["order_id", "order_item_id", "seller_id"]].copy()
shipping_check["shipping_limit_date"] = parsed_dates[("olist_order_items_dataset", "shipping_limit_date")]
order_context = datasets["olist_orders_dataset"][["order_id", "order_status"]].copy()
order_context["purchase_date"] = parsed_dates[("olist_orders_dataset", "order_purchase_timestamp")]
shipping_check = shipping_check.merge(order_context, on="order_id", how="left", validate="many_to_one")
shipping_check["dias_entre_compra_e_limite"] = (
    shipping_check["shipping_limit_date"] - shipping_check["purchase_date"]
).dt.total_seconds() / 86400
display(shipping_check.nlargest(10, "shipping_limit_date"))

,order_id,order_item_id,seller_id,shipping_limit_date,order_status,purchase_date,dias_entre_compra_e_limite
85729,c2bb89b5c1dd978d507284be78a04cb2,1,7a241947449cc45dbfda4f9d0798d9d0,2020-04-09 22:35:08,delivered,2017-05-23 22:28:36,1052.004537
85730,c2bb89b5c1dd978d507284be78a04cb2,2,7a241947449cc45dbfda4f9d0798d9d0,2020-04-09 22:35:08,delivered,2017-05-23 22:28:36,1052.004537
8643,13bdf405f961a6deec817d817f5c6624,1,7a241947449cc45dbfda4f9d0798d9d0,2020-02-05 03:30:51,canceled,2017-03-16 02:30:51,1056.041667
68516,9c94a4ea2f7876660fa6f1b59b69c8e6,1,7a241947449cc45dbfda4f9d0798d9d0,2020-02-03 20:23:22,shipped,2017-03-14 19:23:22,1056.041667
26104,3b61aab5de69abc1731138bd104a777f,1,610f72e407cdd7caaa2f8167b0163fd8,2018-09-18 21:10:15,delivered,2018-08-25 20:59:18,24.007604
54967,7cfdf7265c9572fc7b7cbd3b9cc438b7,2,cee48807215b30a12ca2ca10ffb5f250,2018-09-14 12:30:56,delivered,2018-08-21 12:20:32,24.007222
11891,1afe384f199748cff7a42c9902065560,1,610f72e407cdd7caaa2f8167b0163fd8,2018-09-14 02:09:37,delivered,2018-08-21 01:45:43,24.016597
39543,59eaa904b3f0dbde2785ac1b27eccd18,1,f61c63d13f7cd800549d5acdd390ae72,2018-09-13 14:55:28,delivered,2018-08-20 10:19:46,24.191458
91384,cf5c8d9f52807cb2d2f0a0ff54c478da,6,dfc475d54e1b6dbeeb7d7d9bdaa63827,2018-09-12 13:24:27,delivered,2018-08-24 13:04:05,19.014144
93959,d4fae577806d683110e00e18a5e181be,4,f0b47fbbc6dee9aafe415a6e33051b3f,2018-09-12 03:15:36,delivered,2018-08-28 19:32:05,14.321887


# 13. Cardinalidade das colunas

Cardinalidade é a quantidade de valores distintos de uma coluna. Aqui, `nunique(dropna=False)` inclui o marcador de ausência como um valor distinto. Portanto, uma contagem de 74 categorias pode representar 73 categorias preenchidas e um marcador de ausência.

Essa medida ajuda a identificar identificadores e campos categóricos, mas não substitui a validação das relações 1:1 e 1:N entre tabelas.

In [24]:
cardinality = []

for name, df in datasets.items():

    for column in df.columns:

        cardinality.append({
            "tabela": name,
            "coluna": column,
            "valores_unicos": df[column].nunique(dropna=False),
            "linhas": len(df),
            "percentual_unico": round(
                (
                    df[column].nunique(dropna=False)
                    / len(df)
                ) * 100,
                2
            ) if len(df) else 0
        })

cardinality_df = pd.DataFrame(cardinality)

cardinality_df.sort_values(
    ["percentual_unico", "valores_unicos"],
    ascending=False
).reset_index(drop=True)


,tabela,coluna,valores_unicos,linhas,percentual_unico
0,olist_customers_dataset,customer_id,99441,99441,100.00
1,olist_orders_dataset,order_id,99441,99441,100.00
2,olist_orders_dataset,customer_id,99441,99441,100.00
3,olist_products_dataset,product_id,32951,32951,100.00
4,olist_sellers_dataset,seller_id,3095,3095,100.00
5,product_category_name_translation,product_category_name,71,71,100.00
6,product_category_name_translation,product_category_name_english,71,71,100.00
7,olist_order_reviews_dataset,order_id,98673,99224,99.44
8,olist_orders_dataset,order_purchase_timestamp,98875,99441,99.43
9,olist_order_reviews_dataset,review_id,98410,99224,99.18


# 14. Resumo das variáveis numéricas

`describe()` apresenta contagem de valores preenchidos, média, desvio padrão, mínimo, quartis e máximo.

Prefixos de CEP e identificadores sequenciais foram excluídos: suas médias não têm significado como medidas de negócio. Um máximo distante da média é um sinal para investigação, não evidência suficiente para excluir ou corrigir o registro.

In [25]:
for name, df in datasets.items():
    numeric_cols = [
        col for col in df.select_dtypes(include=np.number).columns
        if not col.endswith("_zip_code_prefix")
        and col not in {"order_item_id", "payment_sequential"}
    ]
    if numeric_cols:
        print(f"RESUMO NUMÉRICO — {name}")
        display(df[numeric_cols].describe().T)

RESUMO NUMÉRICO — olist_geolocation_dataset


,count,mean,std,min,25%,50%,75%,max
geolocation_lat,1000163.0,-21.176153,5.715866,-36.605374,-23.603546,-22.919377,-19.979620,45.065933
geolocation_lng,1000163.0,-46.390541,4.269748,-101.466766,-48.573172,-46.637879,-43.767709,121.105394


RESUMO NUMÉRICO — olist_order_items_dataset


,count,mean,std,min,25%,50%,75%,max
price,112650.0,120.653739,183.633928,0.85,39.90,74.99,134.90,6735.00
freight_value,112650.0,19.990320,15.806405,0.00,13.08,16.26,21.15,409.68


RESUMO NUMÉRICO — olist_order_payments_dataset


,count,mean,std,min,25%,50%,75%,max
payment_installments,103886.0,2.853349,2.687051,0.0,1.00,1.0,4.0000,24.00
payment_value,103886.0,154.100380,217.494064,0.0,56.79,100.0,171.8375,13664.08


RESUMO NUMÉRICO — olist_order_reviews_dataset


,count,mean,std,min,25%,50%,75%,max
review_score,99224.0,4.086421,1.347579,1.0,4.0,5.0,5.0,5.0


RESUMO NUMÉRICO — olist_products_dataset


,count,mean,std,min,25%,50%,75%,max
product_name_lenght,32341.0,48.476949,10.245741,5.0,42.0,51.0,57.0,76.0
product_description_lenght,32341.0,771.495285,635.115225,4.0,339.0,595.0,972.0,3992.0
product_photos_qty,32341.0,2.188986,1.736766,1.0,1.0,1.0,3.0,20.0
product_weight_g,32949.0,2276.472488,4282.038731,0.0,300.0,700.0,1900.0,40425.0
product_length_cm,32949.0,30.815078,16.914458,7.0,18.0,25.0,38.0,105.0
product_height_cm,32949.0,16.937661,13.637554,2.0,8.0,13.0,21.0,105.0
product_width_cm,32949.0,23.196728,12.079047,6.0,15.0,20.0,30.0,118.0


# 15. Validações baseadas em regras de negócio

Por exemplo:

> Uma entrega pode acontecer antes da compra?

Logicamente não.


In [26]:
if orders_name:

    orders_check = datasets[orders_name].copy()

    date_cols = [
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]

    for col in date_cols:

        if col in orders_check.columns:

            orders_check[col] = pd.to_datetime(
                orders_check[col],
                errors="coerce"
            )

    if {
        "order_purchase_timestamp",
        "order_delivered_customer_date"
    }.issubset(orders_check.columns):

        invalid_delivery_dates = (
            orders_check["order_delivered_customer_date"]
            <
            orders_check["order_purchase_timestamp"]
        ).sum()

        print(
            "Pedidos entregues antes da compra:",
            invalid_delivery_dates
        )


Pedidos entregues antes da compra: 0


## 15.1 Validações temporais e de domínio

As comparações temporais abaixo mostram quantos registros possuem as duas datas e quantos violam a sequência esperada. Uma violação é um caso para investigação.

A caixa geográfica usada é uma **triagem aproximada**, não uma fronteira oficial: pontos dentro dela ainda podem estar fora do Brasil. Para uma análise geográfica definitiva, valide os pontos contra a geometria do país.

In [27]:
temporal_records = []
for start, end in [
    ("order_purchase_timestamp", "order_approved_at"),
    ("order_approved_at", "order_delivered_carrier_date"),
    ("order_delivered_carrier_date", "order_delivered_customer_date"),
    ("order_purchase_timestamp", "order_estimated_delivery_date"),
]:
    a = parsed_dates[("olist_orders_dataset", start)]
    b = parsed_dates[("olist_orders_dataset", end)]
    valid = a.notna() & b.notna()
    temporal_records.append({"inicio": start, "fim": end,
                             "comparaveis": int(valid.sum()),
                             "fora_da_sequencia": int((valid & b.lt(a)).sum())})
display(pd.DataFrame(temporal_records))

products = datasets["olist_products_dataset"]
payments = datasets["olist_order_payments_dataset"]
items = datasets["olist_order_items_dataset"]
reviews = datasets["olist_order_reviews_dataset"]
geo = datasets["olist_geolocation_dataset"]
geo_valid = geo[["geolocation_lat", "geolocation_lng"]].notna().all(axis=1)
geo_suspect = geo_valid & ~(
    geo["geolocation_lat"].between(-34, 6) & geo["geolocation_lng"].between(-74, -28)
)
domain_checks = {
    "Produtos com peso zero": products["product_weight_g"].eq(0),
    "Produtos com peso negativo": products["product_weight_g"].lt(0),
    "Produtos com alguma dimensão não positiva": products[["product_length_cm", "product_height_cm", "product_width_cm"]].le(0).any(axis=1),
    "Pagamentos com zero parcelas": payments["payment_installments"].eq(0),
    "Pagamentos com parcelas negativas": payments["payment_installments"].lt(0),
    "Pagamentos de valor zero": payments["payment_value"].eq(0),
    "Pagamentos de valor negativo": payments["payment_value"].lt(0),
    "Itens de preço não positivo": items["price"].le(0),
    "Itens de frete negativo": items["freight_value"].lt(0),
    "Notas preenchidas fora de 1 a 5": reviews["review_score"].notna() & ~reviews["review_score"].isin([1, 2, 3, 4, 5]),
    "Coordenadas fora da caixa aproximada": geo_suspect,
}
display(pd.DataFrame([{"verificacao": label, "linhas_sinalizadas": int(mask.sum())}
                      for label, mask in domain_checks.items()]))
display(payments.loc[payments["payment_installments"].eq(0) | payments["payment_value"].eq(0)]
        .groupby("payment_type", dropna=False)
        .agg(registros=("order_id", "size"), pedidos=("order_id", "nunique")))
display(geo.loc[geo_suspect].head(10))

,inicio,fim,comparaveis,fora_da_sequencia
0,order_purchase_timestamp,order_approved_at,99281,0
1,order_approved_at,order_delivered_carrier_date,97644,1359
2,order_delivered_carrier_date,order_delivered_customer_date,96475,23
3,order_purchase_timestamp,order_estimated_delivery_date,99441,0


,verificacao,linhas_sinalizadas
0,Produtos com peso zero,4
1,Produtos com peso negativo,0
2,Produtos com alguma dimensão não positiva,0
3,Pagamentos com zero parcelas,2
4,Pagamentos com parcelas negativas,0
5,Pagamentos de valor zero,9
6,Pagamentos de valor negativo,0
7,Itens de preço não positivo,0
8,Itens de frete negativo,0
9,Notas preenchidas fora de 1 a 5,0


,registros,pedidos
payment_type,,
credit_card,2,2
not_defined,3,3
voucher,6,5


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
513631,28165,41.614052,-8.411675,vila nova de campos,RJ
513643,28155,-34.586422,-58.732101,santa maria,RJ
513754,28155,42.439286,13.820214,santa maria,RJ
514429,28333,38.381672,-6.328200,raposo,RJ
516682,28595,43.684961,-7.411080,portela,RJ
538512,29654,29.409252,-98.484121,santo antônio do canaã,ES
538557,29654,21.657547,-101.466766,santo antonio do canaa,ES
585242,35179,25.995203,-98.078544,santana do paraíso,MG
585260,35179,25.995245,-98.078533,santana do paraiso,MG


# 16. Modelo inicial dos dados

| Tabela | Unidade de cada linha | Chave / ligação a validar |
|---|---|---|
| `olist_orders_dataset` | Pedido | `order_id`, único e sem nulos nos resultados originais |
| `olist_customers_dataset` | Cadastro de cliente associado a um pedido | `customer_id`; usar `customer_unique_id` para recorrência |
| `olist_order_items_dataset` | Item de pedido | Chave composta (`order_id`, `order_item_id`) |
| `olist_order_payments_dataset` | Registro de pagamento do pedido | Candidata: (`order_id`, `payment_sequential`) |
| `olist_order_reviews_dataset` | Registro de avaliação | Nem `review_id` nem `order_id` são únicos; investigar a chave |
| `olist_products_dataset` | Produto | `product_id` |
| `olist_sellers_dataset` | Vendedor | `seller_id` |
| `olist_geolocation_dataset` | Observação geográfica de prefixo de CEP | Prefixo se repete; definir regra de consolidação antes de juntar |
| `product_category_name_translation` | Tradução de categoria | `product_category_name` |

**Prevenção de duplicação de valores:** itens, pagamentos e avaliações podem ter várias linhas por pedido. Agregue cada tabela no nível de pedido antes de combinar indicadores nesse nível. Use `merge(validate="one_to_one")` ou `merge(validate="many_to_one")`, conforme a relação esperada, e confira contagens e somas antes e depois.

Nesta amostra, `customer_id` é único tanto em clientes quanto em pedidos. A recorrência de consumidores deve ser medida por `customer_unique_id`, que possui 96.096 valores distintos.

A tradução de categorias e a geolocalização também fazem parte do modelo; suas coberturas precisam ser verificadas.

# 17. Registro dos achados e decisões de tratamento

Os números abaixo vêm das saídas salvas no notebook original. As verificações complementares desta revisão ainda devem ser executadas com os CSVs.

| Prioridade | Evidência observada | Interpretação e ação recomendada |
|---|---|---|
| Alta | Oito colunas de data carregadas como `object`; `order_approved_at` não era inspecionada | Converter com auditoria de falhas; incluir aprovação no diagnóstico |
| Alta | `shipping_limit_date` chega a **09/04/2020**, enquanto a última compra é de **17/10/2018** | Investigar pedidos e intervalos envolvidos; não corrigir automaticamente para 2018 |
| Alta | Geolocalização: latitude máxima **45,07** e longitude máxima **121,11** | Há coordenadas incompatíveis com uma localização no Brasil; validar antes de mapas e distâncias |
| Alta | **261.831** duplicatas exatas em geolocalização (**26,18%**) | Definir remoção de duplicatas exatas e consolidação por prefixo separadamente |
| Alta | Itens, pagamentos e avaliações repetem `order_id` | Evitar multiplicação de linhas e valores nas junções |
| Média | **2.965** datas de entrega, **1.783** datas de envio e **160** aprovações ausentes | Cruzar ausência com `order_status`; não preencher datas arbitrariamente |
| Média | **610** ausentes em cada um de quatro atributos de produtos (**1,85%** por coluna) | Verificar se ocorrem nos mesmos registros; não somar as contagens como produtos distintos |
| Média | `product_photos_qty` é `float64` | Validar integralidade e converter para `Int64`, preservando ausentes; desconhecido não equivale a zero fotos |
| Média | Peso mínimo e número mínimo de parcelas iguais a **0** | Quantificar e inspecionar por produto ou tipo de pagamento; causas ainda não demonstradas |
| Média | `payment_value` mínimo **0,00** | Investigar por tipo de pagamento e pedido; não excluir automaticamente |
| Média | Prefixos de CEP são `int64` | Ler como texto e preservar cinco dígitos para junções consistentes |
| Contexto | **87.656** títulos e **58.247** comentários de avaliação ausentes | Preservar notas; ausência de comentário não invalida toda a avaliação |
| Contexto | `payment_sequential` máximo **29** | É um identificador sequencial, não uma medida cuja distância da média prove erro |

Os campos `product_name_lenght` e `product_description_lenght` têm grafia herdada da fonte. Se forem renomeados para `*_length` na camada tratada, documente o mapeamento e atualize todas as referências.

# 18. Conclusões e próxima etapa

## Diagnóstico sustentado pelos resultados originais

O conjunto contém **9 tabelas**, **99.441 pedidos**, **112.650 itens**, **32.951 produtos** e **3.095 vendedores**. Há **96.096 consumidores distintos** em `customer_unique_id`.

As compras registradas vão de **04/09/2016 a 17/10/2018**. Esse é o período de compras, não o intervalo global de todos os eventos: há estimativas de entrega até novembro de 2018 e um prazo de envio em abril de 2020 a investigar.

`order_id` é único e sem nulos em pedidos. A chave composta de itens não apresenta duplicatas; as duas relações originalmente testadas — pedidos → clientes e itens → pedidos — não apresentam chaves órfãs. Isso não permite concluir que todas as relações foram validadas.

O teste original encontrou **zero entregas anteriores à compra**. Ainda é necessário verificar os demais eventos temporais e os campos ausentes por status.

## Condição para avançar

A descoberta fornece uma boa base para iniciar a **limpeza e validação**. Ainda não é uma base pronta para indicadores finais. Execute as células revisadas e complementares, registre seus resultados e defina regras para datas, ausentes, duplicatas, chaves e geolocalização.

Preserve os dados brutos e mantenha um registro de quantas linhas e valores cada tratamento altera. Extremos e ausentes devem ser investigados antes de exclusões ou preenchimentos.

## Análises prioritárias após a limpeza

| Ordem | Pergunta de negócio | Definição e cuidado |
|---|---|---|
| 1 | Como pedidos e valor de mercadorias evoluem por mês? | Contar pedidos únicos; somar `price` dos itens; declarar quais status entram e sinalizar meses parciais |
| 2 | Qual é o ticket médio por pedido? | Agregar itens por pedido antes de calcular a média; apresentar frete separadamente |
| 3 | Onde as entregas atrasam mais? | Pedidos entregues com datas válidas; comparar o dia da entrega ao dia estimado; mostrar taxa e tamanho da amostra por UF |
| 4 | Atrasos estão associados a notas menores? | Definir uma regra para múltiplas avaliações por pedido; comparar distribuições e não atribuir causalidade |
| 5 | Quais categorias e vendedores concentram vendas e problemas? | Comparar valor, volume e atrasos com denominadores explícitos; evitar conclusões com amostras pequenas |
| 6 | Quantos consumidores voltam a comprar? | Usar `customer_unique_id`; considerar a janela de observação e o tempo disponível para recompra |

